In [28]:
import pandas as pd

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix



In [29]:
df=pd.read_csv(r"C:\Users\USER\Desktop\5thsem_OJT\project1_diabeties\data\prosessed\fizan_P1W2_features.csv")

In [30]:

from sklearn.metrics import (confusion_matrix,accuracy_score,recall_score,f1_score,roc_auc_score,precision_score)


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_catogory","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
 x,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)

In [31]:
def score_row(name, pred, scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarm": int(fp)
    }

    if scores is not None:
        r["auc"] = round(roc_auc_score(y_test, scores), 4)

    return r

print("train:",x_train.shape,"test:", x_test.shape)
print("diabetic patients in the test set:", int (np.sum(y_test)), "of", len(y_test))
for nm,m,xt in [("LogisticRegression",logreg,x_test_scaled),
                ("decision_tree(d=3)", decision_tree,x_test),
                ("KNN (k=25)", knn, x_test_scaled)]:
    print(f" {nm:22s} accuracy{accuracy_score(y_test,m.predict(xt)):.4f}")

train: (760, 13) test: (190, 13)
diabetic patients in the test set: 55 of 190
 LogisticRegression     accuracy0.7526
 decision_tree(d=3)     accuracy0.7211
 KNN (k=25)             accuracy0.7474


In [32]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182  # Week 5 logistic regression at threshold 0.30
print("PROJECT METRIC : recall (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR,
      "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


### SVM — Simple Definition

**Support Vector Machine (SVM)** is a machine learning algorithm that **separates data into classes by finding the best boundary (hyperplane)** between them.

* Decision Boundary / Hyperplane: Separates the classes.
* Margin: Distance between the boundary and the closest points.
* Support Vectors: Closest points that help define the boundary.

**Simple example:**

> SVM finds the **best boundary** that separates two groups of data with the **largest possible gap (margin)** between them.


In [33]:
from sklearn.svm import SVC

svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("fitted:", svm_rbf)
print("patients flagged:", int (pred_rbf.sum()),"of",len(pred_rbf))

fitted: SVC(random_state=42)
patients flagged: 19 of 190


In [34]:
row_rbf = score_row("SVC rbc (default)", pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("this model  :", row_rbf["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rbc (default)    0.7368     0.6316  0.2182 0.3243     12      43            7 0.6378

target recall: 0.7 | best so far: 0.6182
this model  : 0.2182
